# Notebook 06: Phân tích Đề xuất Hành động (Prescriptive Analysis)
* **Mục tiêu:** Tổng hợp kết quả từ EDA (02), Diagnostic (04), và Predictive/Models (05) để đưa ra các chiến lược kinh doanh thực tế, phục vụ buổi bảo vệ đồ án tốt nghiệp (Defense).


In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập hiển thị
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Tự động tìm file train.csv trong toàn bộ thư mục hiện tại và các thư mục con
train_path = None
for root, dirs, files in os.walk('.'):
    if 'train.csv' in files:
        train_path = os.path.join(root, 'train.csv')
        break

if train_path:
    print(f"Đã tìm thấy file tại đường dẫn: {train_path}")
    df = pd.read_csv(train_path)
    print(f"Kích thước dữ liệu đầu vào: {df.shape}")
    display(df.head(3))
else:
    raise FileNotFoundError("Không tìm thấy file train.csv trong workspace! Hãy kiểm tra lại xem file đã được tải vào thư mục dự án chưa.")

Đã tìm thấy file tại đường dẫn: .\train.csv
Kích thước dữ liệu đầu vào: (9800, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62


In [8]:
print("=== TỔNG HỢP INSIGHT QUAN TRỌNG ===")

# 1. Phân tích phân phối doanh thu
total_sales = df['Sales'].sum()
top_segment_threshold = df['Sales'].quantile(0.75)
high_value_sales = df[df['Sales'] >= top_segment_threshold]['Sales'].sum()
pct_high_value = (high_value_sales / total_sales) * 100

print(f"- Tổng doanh thu toàn tập dữ liệu: {total_sales:,.2f}")
print(f"- Nhóm khách hàng/giao dịch Top 25% chiếm: {pct_high_value:.2f}% tổng doanh thu.")
print("- Insight: Doanh thu có tính tập trung cao (phân phối lệch phải), đòi hỏi chiến lược chăm sóc phân tầng rõ rệt.\n")

# Phân loại nhóm khách hàng/giao dịch
df['Customer_Segment'] = np.where(
    df['Sales'] >= top_segment_threshold, 
    'High-Value (Trọng điểm)', 
    'Standard/Potential (Tiềm năng)'
)

=== TỔNG HỢP INSIGHT QUAN TRỌNG ===
- Tổng doanh thu toàn tập dữ liệu: 2,261,536.78
- Nhóm khách hàng/giao dịch Top 25% chiếm: 83.30% tổng doanh thu.
- Insight: Doanh thu có tính tập trung cao (phân phối lệch phải), đòi hỏi chiến lược chăm sóc phân tầng rõ rệt.



In [9]:
# Xây dựng bảng khuyến nghị chiến lược dựa trên dữ liệu thực tế
recommendations_data = [
    {
        "ID": "REC-01",
        "Category": "Khách hàng & Doanh thu",
        "Khuyến nghị": "Tập trung 70% ngân sách marketing và CSKH vào nhóm High-Value.",
        "Căn cứ dữ liệu": f"Nhóm Top 25% chiếm tới {pct_high_value:.1f}% tổng doanh thu.",
        "Hành động cụ thể": "Xây dựng chương trình VIP Loyalty, ưu đãi độc quyền để giảm tỷ lệ rời bỏ.",
        "Impact": "High",
        "Effort": "Low"
    },
    {
        "ID": "REC-02",
        "Category": "Quản lý Tồn kho & Sản phẩm",
        "Khuyến nghị": "Tối ưu hóa danh mục các sản phẩm/giao dịch có doanh thu thấp.",
        "Căn cứ dữ liệu": "Các sản phẩm nhóm dưới có giá trị thấp hoặc bị giới hạn bởi outlier capping.",
        "Hành động cụ thể": "Đẩy mạnh flash sale, bundle combo xả hàng tồn kho hoặc ngừng kinh doanh mặt hàng kém hiệu quả.",
        "Impact": "Medium",
        "Effort": "Low"
    },
    {
        "ID": "REC-03",
        "Category": "Vận hành & Khu vực",
        "Khuyến nghị": "Tái phân bổ nguồn lực logistics và quảng cáo theo vùng miền.",
        "Căn cứ dữ liệu": "Sự chênh lệch rõ rệt về doanh thu và tỷ lệ chuyển đổi giữa các khu vực từ EDA (Notebook 02).",
        "Hành động cụ thể": "Tăng cường năng lực vận chuyển tại các khu vực trọng điểm, cắt giảm chi phí ở vùng kém hiệu quả.",
        "Impact": "High",
        "Effort": "High"
    },
    {
        "ID": "REC-04",
        "Category": "Dự báo & Phòng ngừa rủi ro",
        "Khuyến nghị": "Ứng dụng mô hình phân loại (K-NN/SVM) để chủ động kích cầu.",
        "Căn cứ dữ liệu": "Xác suất dự báo từ mô hình giúp nhận diện sớm các phân khúc/khu vực có nguy cơ sụt giảm doanh thu.",
        "Hành động cụ thể": "Gửi voucher tự động cho khách hàng thuộc nhóm có nguy cơ chuyển dịch doanh thu thấp trước khi họ rời bỏ.",
        "Impact": "High",
        "Effort": "Medium"
    },
    {
        "ID": "REC-05",
        "Category": "Chiến lược Giá (Pricing)",
        "Khuyến nghị": "Thiết lập chính sách giá động (Dynamic Pricing) theo khung giờ/phân khúc.",
        "Căn cứ dữ liệu": "Phân phối giá trị giao dịch trung bình và độ lệch chuẩn doanh thu.",
        "Hành động cụ thể": "Áp dụng biểu giá bậc thang để khuyến khích khách hàng nâng cao giá trị giỏ hàng trung bình (AOV).",
        "Impact": "Medium",
        "Effort": "High"
    }
]

df_recs = pd.DataFrame(recommendations_data)
display(df_recs[['ID', 'Category', 'Khuyến nghị', 'Impact', 'Effort']])

,ID,Category,Khuyến nghị,Impact,Effort
0,REC-01,Khách hàng & Doanh thu,Tập trung 70% ngân sách marketing và CSKH vào ...,High,Low
1,REC-02,Quản lý Tồn kho & Sản phẩm,Tối ưu hóa danh mục các sản phẩm/giao dịch có ...,Medium,Low
2,REC-03,Vận hành & Khu vực,Tái phân bổ nguồn lực logistics và quảng cáo t...,High,High
3,REC-04,Dự báo & Phòng ngừa rủi ro,Ứng dụng mô hình phân loại (K-NN/SVM) để chủ đ...,High,Medium
4,REC-05,Chiến lược Giá (Pricing),Thiết lập chính sách giá động (Dynamic Pricing...,Medium,High


In [10]:
# Lọc nhóm Quick Wins (High Impact, Low Effort) để đưa lên slide đầu tiên
quick_wins = df_recs[(df_recs['Impact'] == 'High') & (df_recs['Effort'] == 'Low')]

print("=== NHÓM ƯU TIÊN SỐ 1 (QUICK WINS CHO SLIDE DEFENSE) ===")
for idx, row in quick_wins.iterrows( ):
    print(f"[{row['ID']}] {row['Khuyến nghị']}")
    print(   f"  -> Hành động: {row['Hành động cụ thể']}\n")

=== NHÓM ƯU TIÊN SỐ 1 (QUICK WINS CHO SLIDE DEFENSE) ===
[REC-01] Tập trung 70% ngân sách marketing và CSKH vào nhóm High-Value.
  -> Hành động: Xây dựng chương trình VIP Loyalty, ưu đãi độc quyền để giảm tỷ lệ rời bỏ.



In [ ]:
import os

# Đảm bảo thư mục reports nằm ngay trong thư mục dự án
os.makedirs('reports', exist_ok=True)

# Lưu file kết quả khuyến nghị
df_recs.to_csv('reports/business_recommendations_final.csv', index=False)
print("Đã xuất báo cáo đề xuất thành công ra file: reports/business_recommendations_final.csv")

Đã xuất báo cáo đề xuất thành công ra file: ../reports/business_recommendations_final.csv
